# Image JEPA Training on PROSTATEx

This notebook demonstrates training an Image JEPA (Joint Embedding Predictive Architecture) model on lesion-centered PROSTATEx MRI patches using the `eb_jepa` framework.

## Hardware Requirements

- This notebook is designed to run on Google Colab with a GPU runtime
- Recommended: GPU with at least 12GB VRAM (for example Tesla T4 or P100)
- Training time: preprocessing depends on Drive I/O, training usually takes a few hours depending on GPU and epochs

## Features

- PROSTATEx preprocessing directly from the public training download
- Colab-friendly config generation for Image JEPA training
- Optional Weights & Biases logging integration


In [ ]:
# Check if we're running on GPU
!nvidia-smi


In [ ]:
# Install required packages
!pip install fire omegaconf wandb tqdm pydicom SimpleITK scikit-learn


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive')
REPO_URL = 'https://github.com/DematteisGiacomo/eb_jepa.git'
REPO_BRANCH = 'feat/prostatex'
REPO_DIR = Path('/content/eb_jepa')

RAW_PROSTATEX_DIR = DRIVE_ROOT / 'PROSTATEx'
OUTPUT_DIR = DRIVE_ROOT / 'eb_jepa_data' / 'prostatex'

# Optional overrides if auto-detection is not enough.
IMAGES_CSV = None
FINDINGS_CSV = None
IMAGE_PATTERNS = ('*Images*.csv', '*images*.csv')
FINDING_PATTERNS = ('*Findings*.csv', '*findings*.csv')


def find_matches(root_dir, patterns, limit=10):
    matches = []
    seen = set()
    for pattern in patterns:
        for path in root_dir.rglob(pattern):
            path = path.resolve()
            if path in seen:
                continue
            seen.add(path)
            matches.append(path)
            if len(matches) >= limit:
                return matches
    return matches


print('REPO_URL =', REPO_URL)
print('REPO_BRANCH =', REPO_BRANCH)
print('REPO_DIR =', REPO_DIR)
print('RAW_PROSTATEX_DIR =', RAW_PROSTATEX_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)
print('IMAGES_CSV =', IMAGES_CSV)
print('FINDINGS_CSV =', FINDINGS_CSV)

print('\nTop-level folders in MyDrive:')
for path in sorted(DRIVE_ROOT.iterdir())[:20]:
    print(' -', path)

image_csv_candidates = find_matches(DRIVE_ROOT, IMAGE_PATTERNS, limit=5)
finding_csv_candidates = find_matches(DRIVE_ROOT, FINDING_PATTERNS, limit=5)

print('\nCandidate image CSVs found in Drive:')
for path in image_csv_candidates:
    print(' -', path)

print('\nCandidate finding CSVs found in Drive:')
for path in finding_csv_candidates:
    print(' -', path)

if not image_csv_candidates and not finding_csv_candidates:
    print('\nNo PROSTATEx CSV candidates found yet. Double-check that the dataset is extracted in Drive.')


In [ ]:
# Clone the repository
import os
import shutil

os.chdir('/content')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

!git clone --branch {REPO_BRANCH} --single-branch {REPO_URL}
%cd /content/eb_jepa
!python -m pip uninstall -y torchaudio
!pip install -e .


In [ ]:
# Set environment variables for datasets and checkpoints
%env EBJEPA_DSETS=/content/drive/MyDrive/eb_jepa_data
# %env EBJEPA_CKPTS=/content/drive/MyDrive/eb_jepa_checkpoints


import json
import os
from pathlib import Path

os.chdir(REPO_DIR)
raw_root = Path(RAW_PROSTATEX_DIR)
output_root = Path(OUTPUT_DIR)
output_root.mkdir(parents=True, exist_ok=True)


def detect_first_csv(root_dir, patterns):
    for pattern in patterns:
        matches = sorted(root_dir.rglob(pattern))
        if matches:
            return matches[0]
    return None


if not raw_root.exists():
    print(f'RAW_PROSTATEX_DIR does not exist: {raw_root}')
    print('\nUpdate RAW_PROSTATEX_DIR in the previous cell to the extracted dataset root.')
    print('\nTop-level folders in MyDrive:')
    for path in sorted(DRIVE_ROOT.iterdir())[:20]:
        print(' -', path)
    print('\nCandidate image CSVs found in Drive:')
    for path in find_matches(DRIVE_ROOT, IMAGE_PATTERNS, limit=10):
        print(' -', path)
    print('\nCandidate finding CSVs found in Drive:')
    for path in find_matches(DRIVE_ROOT, FINDING_PATTERNS, limit=10):
        print(' -', path)
    raise FileNotFoundError(
        'Set RAW_PROSTATEX_DIR to the extracted dataset root in Drive, then rerun this cell.'
    )

images_csv = Path(IMAGES_CSV) if IMAGES_CSV else detect_first_csv(raw_root, IMAGE_PATTERNS)
findings_csv = Path(FINDINGS_CSV) if FINDINGS_CSV else detect_first_csv(raw_root, FINDING_PATTERNS)

if images_csv is None or findings_csv is None:
    print('Could not auto-detect the required PROSTATEx CSV files under RAW_PROSTATEX_DIR.')
    print('Top-level contents under RAW_PROSTATEX_DIR:')
    for path in sorted(raw_root.iterdir())[:20]:
        print(' -', path)
    print('\nCSV files discovered recursively under RAW_PROSTATEX_DIR:')
    for path in sorted(raw_root.rglob('*.csv'))[:50]:
        print(' -', path)
    raise FileNotFoundError(
        'Set RAW_PROSTATEX_DIR to the extracted dataset root, or set IMAGES_CSV and FINDINGS_CSV explicitly in the path cell.'
    )

print('Using images_csv =', images_csv)
print('Using findings_csv =', findings_csv)
print('Writing outputs to =', output_root)

!python -m examples.image_jepa.prostatex_preprocess --raw_root "{raw_root}" --output_dir "{output_root}" --images_csv "{images_csv}" --findings_csv "{findings_csv}"

stats_path = output_root / 'dataset_stats.json'
manifest_path = output_root / 'prostatex_manifest.csv'

with stats_path.open() as f:
    stats = json.load(f)

mean = [float(x) for x in stats['mean']]
std = [float(x) for x in stats['std']]

print(json.dumps(stats, indent=2))
print('manifest_path =', manifest_path)
print('mean =', mean)
print('std =', std)


In [ ]:
# Create a modified config for Colab
from pathlib import Path
import json

TRAIN_CONFIG_PATH = REPO_DIR / 'examples' / 'image_jepa' / 'cfgs' / 'prostatex_colab.yaml'

colab_config = f'''meta:
  seed: 42
  device: auto

data:
  dataset: prostatex
  data_dir: {json.dumps(str(OUTPUT_DIR))}
  manifest_path: {json.dumps(str(manifest_path))}
  train_split: train
  val_split: val
  batch_size: 64
  num_workers: 2
  image_size: 224
  in_channels: 3
  num_classes: 2
  transform_profile: medical
  crop_scale: [0.8, 1.0]
  num_crops: 2
  mean: {json.dumps(mean)}
  std: {json.dumps(std)}

model:
  type: resnet
  patch_size: 16
  use_projector: true
  proj_hidden_dim: 1024
  proj_output_dim: 1024

loss:
  type: vicreg
  std_coeff: 1.0
  cov_coeff: 25.0
  lmbd: 10.0

optim:
  epochs: 50
  lr: 0.2
  weight_decay: 1.0e-4
  warmup_epochs: 5
  warmup_start_lr: 3.0e-5
  min_lr: 0.0

logging:
  log_wandb: false
  log_every: 1
  save_every: 10
  tqdm_silent: false

training:
  use_amp: true
  dtype: bfloat16
'''

with TRAIN_CONFIG_PATH.open('w') as f:
    f.write(colab_config)

print(f'Wrote {TRAIN_CONFIG_PATH}')
print(colab_config)

In [ ]:
# Optional: Configure W&B logging
import wandb
# wandb.login()  # Uncomment to use W&B logging

In [ ]:
# Start training
!python -m examples.image_jepa.main --fname examples/image_jepa/cfgs/prostatex_colab.yaml

## Notes

- Extract the public PROSTATEx training download somewhere under `RAW_PROSTATEX_DIR` before running the preprocessing cell.
- The notebook writes `examples/image_jepa/cfgs/prostatex_colab.yaml` using the computed manifest path and normalization stats.
- If you want W&B logging, run `wandb.login()` and change `log_wandb: false` to `log_wandb: true` in the generated config cell.
- To run longer training, increase `optim.epochs` or projector dimensions in the Colab config cell.